# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MisbahSangi/flyrank-ml-internship-misbah/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Refresh / Content Opportunity Scoring**

I'm choosing this lane because the Week 1–3 notebooks I've already run are direct
examples of it: I've seen a hand-written baseline rule (stale_visible_page,
declining_with_demand) scored against a learned model, and watched the model beat
the rule roughly 3x on Precision@50 (0.240 -> 0.740). This lane asks the same
question I've been practicing: which pages deserve a human reviewer's limited time
first? I want to extend this into a stronger, future-looking version — instead of
"is this page currently down" (the starter's proxy label), I want to test
"given the prior 90 days, does this page decline over the next 30" using the full
warehouse release.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision this improves:** which content pages a review/content team looks at
first, out of far more candidates than they have time to review.

**Who acts on it:** a content or SEO reviewer with limited weekly capacity —
say, capacity to review 50 pages, not 30,000.

**The action:** based on the ranked queue and reason codes, the reviewer refreshes,
expands, protects, prunes, or monitors a given page.

**Unit of analysis:** one row = one content item (content_hash_id) at a given
scoring point in time, not a client and not a single day.

**Cost of a wrong call:**
- False positive (flagged as needing review, but it wasn't actually declining):
  wastes reviewer time on a page that didn't need it — capacity is finite, so
  every wasted review is a real page that should have been reviewed instead.
- False negative (a genuinely declining page never surfaces in the top-K queue):
  worse — a real problem goes unnoticed until it's more costly to fix, and the
  team never even sees it to make a judgment call.

**Why data/ML can help at all:** with 30,000+ pages (and 500K+ in the full
warehouse) and a review team that can only look at a fraction of them, a hand
rule can only encode a few "if/else" conditions a person thought of in advance.
A model can weigh many observed signals (staleness, visibility, position,
momentum, query concentration) together and rank consistently at scale — the
starter results already show this beating the hand rule by ~3x at Precision@50.
This is decision-support, not proof that refreshing a page will cause it to
recover — that would need a causal experiment this data can't give me.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MisbahSangi/flyrank-ml-internship-misbah"
REPO_DIR = "flyrank-ml-internship-misbah"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-misbah
Starter data found. You're ready.


In [4]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} rows, {df.shape[1]} columns")

# Number 1: how many pages are currently flagged 'declining' (the starter's proxy label)
declining_rate = (df["trend_direction"] == "down").mean()
print(f"\n1) Share of pages currently labeled 'declining': {declining_rate:.1%}")

# Number 2: how many pages match the starter's clearest baseline reason code
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
print(f"2) Pages matching 'stale_visible_page' (stale AND still getting real traffic): {stale_visible:,} "
      f"({stale_visible/len(df):.1%} of all pages)")

# Number 3: the already-verified model comparison from Week 1's run
print("3) From outputs/model_results.json (already run in notebook 01):")
print("   Hand-rule baseline  Precision@50: 0.240 (~12 of top 50 correct)")
print("   Random forest       Precision@50: 0.740 (~37 of top 50 correct)")
print("   -> learned ranking beat the fixed rule ~3.1x on this starter slice")

30,000 rows, 44 columns

1) Share of pages currently labeled 'declining': 54.2%
2) Pages matching 'stale_visible_page' (stale AND still getting real traffic): 17 (0.1% of all pages)
3) From outputs/model_results.json (already run in notebook 01):
   Hand-rule baseline  Precision@50: 0.240 (~12 of top 50 correct)
   Random forest       Precision@50: 0.740 (~37 of top 50 correct)
   -> learned ranking beat the fixed rule ~3.1x on this starter slice


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**I can claim (observed / directional / decision-support):**
- Which observed signals (staleness, visibility, position, momentum) are
  *associated with* pages later trending down, based on this dataset.
- A ranked review queue that plausibly puts more genuinely-declining pages near
  the top than a fixed hand rule does, measured by Precision@K on held-out data.
- That the starter's learned model outperforms its hand-written baseline on this
  specific slice, under client-holdout validation.

**I cannot claim:**
- That I proved anything about Google's ranking algorithm.
- That refreshing a flagged page will cause it to recover — that needs a real
  experiment (e.g. A/B test on actual refreshes), not this observational data.
- That "declining" as currently defined (trend_direction == down, a same-window
  bucket) is the same as a true future decline — it's a proxy label, and a
  stronger version of this lane should predict a genuine future-window outcome
  instead (prior 90 days -> next 30 days), per the lane guide's own warning.
- Anything about specific real clients, URLs, or queries — everything here is
  pseudonymized and stays that way in anything I publish.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.